# Lab 11 — Độ cao và đất / Elevation and soil

**Viễn thám Nông nghiệp & Môi trường — Chương 11** · [Mở chương / Open chapter](https://quanta1985.github.io/rs-agri/vi/soil-terrain/)

1. Bao nhiêu đất nằm dưới 1 m, 2 m: DeltaDTM (mặt đất) so với Copernicus DEM (bề mặt) / How much land lies below 1 m and 2 m: DeltaDTM (ground) against the Copernicus DEM (surface).
2. Đường cong diện tích – độ cao / The hypsometric curve.
3. DSM trừ DTM ở vườn dừa Bến Tre / DSM minus DTM in the Ben Tre coconut groves.
4. Mốc độ cao: so với cái gì? / The vertical datum: above what?
5. pH đất SoilGrids và đất phèn / SoilGrids pH and acid sulfate soils.

> Ở châu thổ cao một, hai mét, sai số vài mét của mô hình độ cao là toàn bộ câu chuyện.
> In a delta one or two metres high, an elevation error of a few metres is the whole story.

In [ ]:
import ee
import numpy as np
import matplotlib.pyplot as plt

PROJECT = 'YOUR-CLOUD-PROJECT-ID'  # <-- sửa / edit
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

def pixels(img, crs, x0, y1, n_x, n_y, res):
    """Tải ảnh về numpy trên một lưới cố định / fetch an image as numpy on a fixed grid (upper-left x0, y1)."""
    arr = ee.data.computePixels({'expression': img, 'fileFormat': 'NUMPY_NDARRAY',
        'grid': {'dimensions': {'width': n_x, 'height': n_y}, 'crsCode': crs,
                 'affineTransform': {'scaleX': res, 'shearX': 0, 'translateX': x0, 'shearY': 0, 'scaleY': -res, 'translateY': y1}}})
    return {b: arr[b] for b in arr.dtype.names}

# Bản geoBoundaries trong Earth Engine mất các chữ ngoài bảng Latin-1 ('Hậu Giang' -> 'H?u Giang'): đổi tên theo cùng cách.
# The Earth Engine copy of geoBoundaries lost the letters outside Latin-1 ('Hậu Giang' -> 'H?u Giang'): convert names the same way.
def gb_name(s):
    return s.encode('latin-1', 'replace').decode('latin-1')

GB = ee.FeatureCollection('WM/geoLab/geoBoundaries/600/ADM1').filter(ee.Filter.eq('shapeGroup', 'VNM'))

def provinces(names):
    """Tỉnh theo tên, khớp cả tên đúng lẫn tên bị lỗi / provinces by name, matching the correct and the corrupted spelling."""
    fc = GB.filter(ee.Filter.inList('shapeName', list(names) + [gb_name(n) for n in names]))
    n = fc.size().getInfo()
    assert n == len(names), f'tìm thấy / found {n} of {len(names)} provinces'
    return fc

PROVINCES = {
    'mekong': ['Long An', 'Tiền Giang', 'Bến Tre', 'Trà Vinh', 'Vĩnh Long', 'Đồng Tháp', 'An Giang', 'Kiên Giang',
               'Cần Thơ', 'Hậu Giang', 'Sóc Trăng', 'Bạc Liêu', 'Cà Mau'],
    'red': ['Hà Nội', 'Vĩnh Phúc', 'Bắc Ninh', 'Quảng Ninh', 'Hải Dương', 'Hải Phòng', 'Hưng Yên', 'Thái Bình',
            'Hà Nam', 'Nam Định', 'Ninh Bình'],
}
NAME = {'mekong': 'ĐBSCL / Mekong', 'red': 'ĐBSH / Red River'}
REGION = {k: provinces(v).geometry() for k, v in PROVINCES.items()}

## 1. Đất dưới 1 m và 2 m / Land below 1 m and 2 m

- **DeltaDTM** v1 (Pronk và cs. 2024, Deltares, CC BY 4.0): mặt đất trần, ~30 m, mét trên geoid EGM2008, mặt nước bị che, giá trị cắt ở 10 m. / bare ground, ~30 m, metres above the EGM2008 geoid, water masked, clipped at 10 m.
- **Copernicus DEM** GLO-30 (bản 2024_1): bề mặt đầu tiên radar thấy, gồm tán cây và mái nhà. / the first surface the radar sees, tree crowns and roofs included.

Cả hai được so trên **cùng các ô** DeltaDTM có dữ liệu. Diện tích được cộng theo từng nấc 0,1 m. Chương 11 cộng ở 30 m; ở đây mặc định `SCALE = 90` m để chạy trong khoảng 20 giây (Earth Engine đọc lớp ảnh thu nhỏ, mỗi ô 90 m là trung bình của các ô 30 m).

Both are compared on **the same cells** that DeltaDTM covers. Areas are summed in 0.1 m steps. Chapter 11 sums at 30 m; here the default is `SCALE = 90` m so it runs in about 20 seconds (Earth Engine reads a reduced pyramid level, each 90 m cell being the mean of the 30 m cells).

In [ ]:
SCALE = 90          # m; 30 = như chương / as in the chapter (khoảng 2 phút / about 2 minutes)
LO, HI = -30, 100   # nấc 0,1 m / 0.1 m steps, -3 ... 10 m
DTM = ee.Image('projects/sat-io/open-datasets/DELTARES/deltadtm_v1').rename('h')
COP_ID = 'COPERNICUS/DEM/GLO30_2024_1'
COP = (ee.ImageCollection(COP_ID).select('DEM').mosaic()
       .setDefaultProjection(ee.ImageCollection(COP_ID).first().select('DEM').projection()).rename('h'))
DEMS = {'DeltaDTM': DTM, 'Copernicus': COP.updateMask(DTM.mask())}   # cùng ô đất / same land cells

def below_curve(img, geom):
    """Diện tích (ha) theo nấc 0,1 m / area per 0.1 m step -> cumulative share below each step."""
    step = img.multiply(10).floor().clamp(LO, HI).int().rename('bin')
    area = ee.Image.pixelArea().divide(1e4).rename('ha').updateMask(img.mask())
    res = area.addBands(step).reduceRegion(ee.Reducer.sum().group(1, 'bin'), geom, SCALE, maxPixels=1e11, tileScale=8).getInfo()
    groups = {g['bin']: g['sum'] for g in res['groups']}
    per = np.array([groups.get(i, 0.0) for i in range(LO, HI + 1)])
    return per.sum(), np.cumsum(per) / per.sum()    # below[i] = share with elevation < (LO + i + 1) / 10 m

LEVELS = np.arange(LO + 1, HI + 2) / 10            # mức trên của mỗi nấc / upper edge of each step, m
CURVES = {}
for k, geom in REGION.items():
    for name, img in DEMS.items():
        total, below = below_curve(img, geom)
        CURVES[k, name] = below
        at = lambda h: 100 * below[np.searchsorted(LEVELS, h - 1e-9)]
        print(f'{NAME[k]:18s} {name:10s}: dưới / below 1 m {at(1):4.1f} %, 2 m {at(2):4.1f} %, 3 m {at(3):4.1f} %; '
              f'diện tích tính / area used {total / 1e6:.2f} triệu / million ha')

Chương 11 (30 m, ranh vùng bản đơn giản hóa của trang, 3,48 triệu ha ĐBSCL, 1,90 triệu ha ĐBSH):

| | DeltaDTM < 1 m | DeltaDTM < 2 m | Copernicus < 2 m |
|---|---|---|---|
| ĐBSCL / Mekong | 4,0 % | 60,0 % | 37,5 % |
| ĐBSH / Red River | 1,0 % | 10,9 % | 10,3 % |

Lần chạy kiểm tra ở 90 m cho ĐBSCL 3,0 % / 61,5 % / 37,5 % và ĐBSH 0,6 % / 10,0 % / 9,5 %; với `SCALE = 30` được 4,0 % / 60,4 % / 38,1 % và 0,8 % / 10,2 % / 9,8 %. Ở 90 m, mỗi ô là trung bình của 9 ô 30 m, nên các ô rất thấp (< 1 m) bị làm mượt và tỷ lệ dưới 1 m giảm. Diện tích tính ở đây khoảng 3,41 và 1,84 triệu ha, nhỏ hơn chương 2–3 % vì ranh geoBoundaries đầy đủ khác ranh đã đơn giản hóa của trang ở bờ biển và đảo.

Chapter 11 (30 m, the site's simplified region outline, 3.48 million ha in the Mekong, 1.90 million ha in the Red River Delta): see the table. The test run at 90 m gave Mekong 3.0 % / 61.5 % / 37.5 % and Red 0.6 % / 10.0 % / 9.5 %; with `SCALE = 30`, 4.0 % / 60.4 % / 38.1 % and 0.8 % / 10.2 % / 9.8 %. At 90 m each cell is the mean of nine 30 m cells, so the very low cells (< 1 m) are smoothed and the share below 1 m drops. The area used here is about 3.41 and 1.84 million ha, 2–3 % less than the chapter's, because the full geoBoundaries lines differ from the site's simplified outline along the coast and around islands.

## 2. Đường cong diện tích – độ cao / Hypsometric curve

Đọc đường cong ở một mức nước là ra tỷ lệ đất thấp hơn mức đó. Đường Copernicus nằm bên phải vì nó đo cả cây và nhà. / Read the curve at a water level to get the share of land below it. The Copernicus curve lies to the right because it also measures trees and houses.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
STYLE = {'DeltaDTM': '-', 'Copernicus': '--'}
COLOR = {'mekong': '#0072b2', 'red': '#d55e00'}
for (k, name), below in CURVES.items():
    ax.plot(LEVELS, 100 * below, STYLE[name], color=COLOR[k], label=f'{NAME[k]} · {name}')
for h in (1, 2):
    ax.axvline(h, color='grey', lw=0.8, ls=':')
ax.set_xlim(-1, 10); ax.set_ylim(0, 100)
ax.set_xlabel('Độ cao, m trên EGM2008 / elevation, m above EGM2008'); ax.set_ylabel('% diện tích thấp hơn / % of area below')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title('Đường cong diện tích – độ cao / hypsometric curve')
plt.show()

## 3. DSM trừ DTM ở Bến Tre / DSM minus DTM at Ben Tre

Khung 10,24 km quanh Giồng Trôm – Ba Tri (106,5° E, 10,1° N), lưới UTM 48N 30 m. Lớp phủ ESA WorldCover 2021 (ô 10 m gần nhất) cho biết ô nào là cây, ruộng hay nhà.

The 10.24 km frame around Giong Trom – Ba Tri (106.5° E, 10.1° N), UTM 48N 30 m grid. ESA WorldCover 2021 (nearest 10 m cell) tells which cells are trees, cropland or houses.

In [ ]:
LON, LAT = 106.50, 10.10                                    # Giồng Trôm (Bến Tre)
N, RES = 341, 30
CX, CY = ee.Geometry.Point([LON, LAT]).transform('EPSG:32648', 1).coordinates().getInfo()   # UTM 48N
X0, Y1 = round(CX) - N * RES // 2, round(CY) + N * RES // 2
wc = ee.ImageCollection('ESA/WorldCover/v200').first().rename('lc')
px = pixels(ee.Image.cat([DTM.rename('dtm'), COP.rename('cop'), wc]).unmask(-999), 'EPSG:32648', X0, Y1, N, N, RES)
ok = (px['dtm'] > -998) & (px['cop'] > -998)
dtm = np.where(ok, px['dtm'], np.nan); cop = np.where(ok, px['cop'], np.nan); diff = cop - dtm
print(f'Khung / frame: DeltaDTM trung vị / median {np.nanmedian(dtm):.2f} m, Copernicus {np.nanmedian(cop):.2f} m')
for code, label in ((10, 'cây gỗ / trees'), (40, 'đất trồng trọt / cropland'), (50, 'xây dựng / built')):
    m = ok & (px['lc'] == code)
    print(f'  {label:26s}: {100 * m.sum() / ok.sum():5.1f} % ô / of cells, Copernicus − DeltaDTM trung vị / median {np.nanmedian(diff[m]):.2f} m')

# trung bình đĩa 1 km như chương / 1 km disc mean as in the chapter
pt = ee.Geometry.Point([LON, LAT]).buffer(1000)
v = ee.Image.cat([DTM.rename('dtm'), COP.rename('cop')]).reduceRegion(ee.Reducer.mean(), pt, 30).getInfo()
print(f"({LON}, {LAT}), đĩa 1 km / 1 km disc: DeltaDTM {v['dtm']:.2f} m, Copernicus {v['cop']:.2f} m, hiệu / difference {v['cop'] - v['dtm']:.2f} m")

fig, ax = plt.subplots(1, 3, figsize=(16, 5), layout='constrained')
for a, arr, title, cmap, rng in ((ax[0], dtm, 'DeltaDTM (mặt đất / ground)', 'viridis', (0, 6)),
                                 (ax[1], cop, 'Copernicus DEM (bề mặt / surface)', 'viridis', (0, 6)),
                                 (ax[2], diff, 'Copernicus − DeltaDTM', 'YlOrRd', (0, 8))):
    im = a.imshow(arr, cmap=cmap, vmin=rng[0], vmax=rng[1]); a.set_title(title); a.set_axis_off()
    fig.colorbar(im, ax=a, shrink=0.7, label='m')
plt.show()

Chương 11 (đĩa 1 km): DeltaDTM 1,78 m, Copernicus 6,03 m, hiệu 4,25 m; lần chạy kiểm tra cho đúng các số đó. Trong cả khung, cây gỗ chiếm 77 % số ô với hiệu trung vị 3,35 m, còn trên đất trồng trọt hiệu chỉ 0,22 m. Các dải đỏ là **giồng cát** trồng dừa và vườn cây ăn trái, có làng xen giữa; ruộng lúa ở giữa các giồng thì hai mô hình gần nhau. Ở ĐBSCL, theo chương, hiệu là 2,38 m ở Cần Thơ và 0,74 m ở ruộng lúa Tháp Mười.

Chapter 11 (1 km disc): DeltaDTM 1.78 m, Copernicus 6.03 m, difference 4.25 m; the test run gave the same numbers. Across the frame, trees cover 77 % of the cells with a median difference of 3.35 m, while on cropland the difference is only 0.22 m. The red bands are **sand ridges** with coconut groves and orchards, villages in between; on the rice fields between the ridges the two models are close. Elsewhere in the Mekong Delta, the chapter gives a difference of 2.38 m at Can Tho and 0.74 m on the Thap Muoi rice fields.

## 4. Mốc độ cao / The vertical datum

Cả hai DEM tính **mét trên geoid EGM2008**. Bản đồ địa hình Việt Nam tính từ mực nước biển trung bình tại trạm **Hòn Dấu** (Hải Phòng). Minderhoud và cs. (2019) dựng DEM ĐBSCL từ số liệu đo đạc quốc gia và được độ cao trung bình 0,82 m trên mốc Hòn Dấu; trung vị của DeltaDTM trên cùng vùng (Chương 11) là khoảng 1,9 m. Khoảng 1 m chênh lệch ấy vừa do mốc khác nhau, vừa do sai số còn lại của DeltaDTM (sai số tuyệt đối trung bình 0,45 m ở vùng ven biển thấp, Pronk và cs. 2024); không tách được hai phần chỉ từ hai DEM này.

**Bài học:** trước khi so một DEM với mực nước, mực triều hay kịch bản nước biển dâng, phải đưa chúng về **cùng một mốc**. Đường cong ở mục 2 đo mức **phơi nhiễm**, không phải diện tích chắc chắn ngập: đê, cống và bơm giữ nước ở ngoài.

Both DEMs are in **metres above the EGM2008 geoid**. Vietnamese topographic maps are referenced to mean sea level at the **Hon Dau** tide gauge (Hai Phong). Minderhoud et al. (2019) built a Mekong Delta DEM from national survey data and found a mean elevation of 0.82 m above Hon Dau; the DeltaDTM median over the same region (Chapter 11) is about 1.9 m. That gap of about 1 m is partly the datum and partly the remaining error of DeltaDTM (mean absolute error 0.45 m in low coastal areas, Pronk et al. 2024); the two DEMs alone cannot separate them.

**Lesson:** before comparing a DEM with water levels, tides or sea-level scenarios, bring them to **the same datum**. The curve in section 2 measures **exposure**, not land that will surely flood: dikes, sluices and pumps keep water out.

## 5. pH đất SoilGrids / SoilGrids soil pH

SoilGrids 2.0 (Poggio và cs. 2021, ISRIC, CC BY 4.0), pH trong nước × 10, trung bình có trọng số các tầng 0–5, 5–15, 15–30 cm (5 : 10 : 15), ở 250 m.

SoilGrids 2.0 (Poggio et al. 2021, ISRIC, CC BY 4.0), pH in water × 10, thickness-weighted mean of the 0–5, 5–15, 15–30 cm layers (5 : 10 : 15), at 250 m.

In [ ]:
sg = ee.Image('projects/soilgrids-isric/phh2o_mean')
ph = ee.Image.cat([sg.select(f'phh2o_{d}_mean').multiply(w / 30) for d, w in (('0-5cm', 5), ('5-15cm', 10), ('15-30cm', 15))]
                  ).reduce('sum').multiply(0.1).rename('ph')
g = REGION['mekong']
st = ee.Image.cat([ph, ph.lt(5.5).rename('lt55'), ph.lt(4.5).rename('lt45')]).reduceRegion(
    ee.Reducer.mean().combine(ee.Reducer.median(), '', True).combine(ee.Reducer.min(), '', True), g, 250, maxPixels=1e10).getInfo()
print(f"ĐBSCL pH 0–30 cm: trung vị / median {st['ph_median']:.2f}, thấp nhất / min {st['ph_min']:.2f}; "
      f"pH < 5,5 trên / on {100 * st['lt55_mean']:.1f} %, pH < 4,5 trên / on {100 * st['lt45_mean']:.2f} % diện tích / of the area")

STEP = 0.0025
x0, y0, x1, y1 = 104.4, 8.5, 106.85, 11.05
mp = pixels(ph.clip(g).unmask(-1), 'EPSG:4326', x0, y1, round((x1 - x0) / STEP), round((y1 - y0) / STEP), STEP)
arr = np.where(mp['ph'] > 0, mp['ph'], np.nan)
plt.figure(figsize=(7, 7))
plt.imshow(arr, cmap='BrBG', vmin=4.5, vmax=7.5, extent=[x0, x1, y0, y1])
plt.colorbar(label='pH (H₂O)', shrink=0.7)
plt.plot(105.83, 10.55, 'k^'); plt.annotate(' Tháp Mười', (105.83, 10.55))
plt.title('pH đất 0–30 cm theo SoilGrids / topsoil pH (SoilGrids)')
plt.show()

Chương 11: trung vị 5,8, thấp nhất 4,8 (vùng trũng Đồng Tháp Mười), 11,7 % diện tích dưới 5,5 và không ô nào dưới 4,5. Lần chạy kiểm tra: trung vị 5,77, thấp nhất 4,85, 13,3 % dưới 5,5, 0,00 % dưới 4,5. Tỷ lệ dưới 5,5 cao hơn chương vì hai lẽ: ranh ở đây gồm cả đảo Phú Quốc, còn bản đồ của chương chỉ phủ khung đất liền (cắt theo khung đó, lần chạy thử cho 12,3 %); và lớp của chương lưu pH theo nấc 0,05, nên giá trị như 5,48 thành 5,50 và không còn "dưới 5,5". Trong khi đó Husson (1998) ước tính ĐBSCL có khoảng 1,6 triệu ha đất phèn, riêng Đồng Tháp Mười 0,4 triệu ha; khi pyrit bị oxy hóa, pH đất phèn xuống dưới 4 (mục 11.2).

SoilGrids đặt **đúng chỗ** chua nhất, nhưng **làm mượt** các giá trị cực đoan: mô hình học từ rất ít phẫu diện ở châu thổ. Dùng nó để có cái nhìn đầu tiên, không để khoanh đất phèn; muốn biết ô ruộng nào phèn phải khoan và đo pH.

Chapter 11: median 5.8, minimum 4.8 (the Plain of Reeds depression), 11.7 % of the area below 5.5 and no cell below 4.5. Test run: median 5.77, minimum 4.85, 13.3 % below 5.5, 0.00 % below 4.5. The share below 5.5 is higher than the chapter's for two reasons: the outline here includes Phu Quoc island, while the chapter's map covers only the mainland box (cut to that box, a trial run gave 12.3 %); and the chapter's layer stores pH in 0.05 steps, so a value such as 5.48 becomes 5.50 and is no longer "below 5.5". Meanwhile Husson (1998) estimated about 1.6 million ha of acid sulfate soils in the Mekong Delta, 0.4 million ha in the Plain of Reeds alone; once the pyrite oxidises, their pH falls below 4 (section 11.2).

SoilGrids puts the most acid area **in the right place**, but **smooths away** the extremes: the model learns from very few profiles in the delta. Use it for a first look, not to map acid sulfate soils; to know which field is acid, drill and measure pH.

## Bài tập / Exercises

1. Đặt `SCALE = 30` và chạy lại mục 1. Tỷ lệ dưới 1 m của ĐBSCL thay đổi thế nào, và vì sao ô thô hơn làm mất đất thấp? / Set `SCALE = 30` and rerun section 1. How does the Mekong share below 1 m change, and why do coarser cells lose low land?
2. Ở mục 3, dời khung đến ruộng lúa Tháp Mười (`LON, LAT = 105.83, 10.55`). Hiệu Copernicus − DeltaDTM trên đất trồng trọt còn bao nhiêu (chương, đĩa 1 km: 0,74 m)? / In section 3, move the frame to the Thap Muoi rice fields (`LON, LAT = 105.83, 10.55`). What is the Copernicus − DeltaDTM difference on cropland now (chapter, 1 km disc: 0.74 m)?
3. Hạ mọi độ cao DeltaDTM xuống 0,275 m (sụt lún 1,1 cm/năm trong 25 năm, mục 11.5: `DTM.subtract(0.275)`) rồi đọc lại tỷ lệ ĐBSCL dưới 2 m. / Lower every DeltaDTM elevation by 0.275 m (25 years of 1.1 cm/yr subsidence, section 11.5: `DTM.subtract(0.275)`) and read the Mekong share below 2 m again.

*DeltaDTM v1.0 (Pronk et al. 2024, Deltares), CC BY 4.0. Copernicus DEM GLO-30 (2024_1): produced using Copernicus WorldDEM-30 © DLR e.V. 2010–2014 and © Airbus Defence and Space GmbH 2014–2018 provided under COPERNICUS by the European Union and ESA. SoilGrids 2.0 (Poggio et al. 2021, ISRIC), CC BY 4.0. ESA WorldCover 10 m 2021 v200, CC BY 4.0. geoBoundaries (Runfola et al. 2020), CC BY 4.0.*